In [ ]:
import pandas as pd

# === Load CSV files ===
file1_path = "./data/LLaVA-Med/file1.csv"
file2_path = "./data/LLaVA-Med/file2.csv"

df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path)

# === Clean Data ===
# Remove invalid or 'not mentioned' entries
df1 = df1[df1['Drug Name'].astype(str).str.lower().ne('not mentioned')]

df2 = df2[
    df2['drug_name'].notna() &
    df2['drug_name'].astype(str).str.strip().ne('') &
    df2['generated_summary'].notna() &
    df2['generated_summary'].astype(str).str.strip().ne('') &
    df2['generated_summary'].astype(str).str.lower().ne('none')
]


# === Initialize ===
matches = []
file2_index = 0

# === Iterate through file1 records ===
for i, row1 in df1.iterrows():
    drug1 = str(row1['Drug Name']).strip().lower()
    found = False

    # Compare with next 5 records of file2 from the current index
    for j in range(file2_index, min(file2_index + 10, len(df2))):
        drug2 = str(df2.iloc[j]['drug_name']).strip().lower()
        if drug1 == drug2:
            matches.append({
                "Preprocessed Posts": row1["Preprocessed Posts"],
                "Drug Name": row1["Drug Name"],
                "Drug Name_FILE2": drug2,
                "generated_summary": df2.iloc[j]["generated_summary"]
            })
            file2_index = j + 1  # update index for next iteration
            found = True
            break

    # Stop after collecting 50 matching records
    if len(matches) >= 50:
        break

# === Create Final DataFrame ===
df3 = pd.DataFrame(matches)

# === Save Output ===
output_path = "./data/LLaVA-Med/file3.csv"
df3.to_csv(output_path, index=False)

print(f"✅ File3 created successfully with {len(df3)} records -> {output_path}")
